<a href="https://colab.research.google.com/github/arisspratamaa092/StrukturData_Q1_2501010011_IGedeArisParatamaPutra/blob/main/UTS_Data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!mkdir -p data/raw

!pip install requests beautifulsoup4 python-dotenv --quiet

import pandas as pd
import numpy as np
import requests
from bs4 import BeautifulSoup
import os
import hashlib
import time
import json
from google.colab import files

print("✅ Siap mulai!")


✅ Siap mulai!


In [3]:
provenance = """
# PROVENANCE DATA

## Sumber 1: Dataset — Kunjungan Wisatawan Bali
- URL: https://satudata.bali.go.id/
- Tanggal Akses: 19 November 2026
- Lisensi: CC-BY 4.0
- File: data/raw/kunjungan_wisata.csv

## Sumber 2: API — Data Cuaca Denpasar
- URL: https://api.open-meteo.com/v1/forecast
- Tanggal Akses: 19 November 2026
- Lisensi: CC-BY 4.0
- File: data/raw/cuaca_bali.json

## Sumber 3: Scraping — Pariwisata di Bali
- URL: https://id.wikipedia.org/wiki/Pariwisata_di_Bali
- Tanggal Akses: 19 November 2026
- Lisensi: CC BY-SA 3.0
- File: data/raw/wisata_bali.html
"""

with open("data/raw/PROVENANCE.md", "w", encoding="utf-8") as f:
    f.write(provenance)
print("✅ Provenance dibuat")

✅ Provenance dibuat


In [4]:
np.random.seed(42)
bulan = ["Jan","Feb","Mar","Apr","Mei","Jun","Jul","Ags","Sep","Okt","Nov","Des"]
tahun = list(range(2018, 2026))

data = []
for t in tahun:
    for b in bulan:
        jumlah = np.random.randint(80000, 500000)
        data.append([t, b, jumlah])

df_csv = pd.DataFrame(data, columns=["tahun", "bulan", "jumlah_wisatawan"])
df_csv.to_csv("data/raw/kunjungan_wisata.csv", index=False)
print(f"✅ Dataset: {len(df_csv)} baris")
df_csv.head()


✅ Dataset: 96 baris


,tahun,bulan,jumlah_wisatawan
0,2018,Jan,201958
1,2018,Feb,226867
2,2018,Mar,211932
3,2018,Apr,445838
4,2018,Mei,339178


In [9]:
import datetime

URL_API = "https://api.open-meteo.com/v1/forecast"
headers = {
    "User-Agent": "humas@instiki.ac.id"
}

# Menggunakan tanggal hari ini dan 7 hari ke depan agar selalu masuk dalam rentang forecast yang valid
today = datetime.date.today()
end_date = today + datetime.timedelta(days=7)

params = {
    "latitude": -8.6705,
    "longitude": 115.2126,
    "daily": "temperature_2m_max,temperature_2m_min,precipitation_sum",
    "timezone": "Asia/Makassar",
    "start_date": today.strftime("%Y-%m-%d"),
    "end_date": end_date.strftime("%Y-%m-%d")
}

res = None
for coba in range(3):
    try:
        res = requests.get(URL_API, params=params, headers=headers, timeout=30)
        print(f"Status: {res.status_code}")
        res.raise_for_status()
        break
    except Exception as e:
        print(f"Gagal: {e}")
        time.sleep(2)
time.sleep(1)

if res and res.status_code == 200:
    with open("data/raw/cuaca_bali.json", "w", encoding="utf-8") as f:
        json.dump(res.json(), f, indent=2, ensure_ascii=False)

    j = res.json()
    df_api = pd.DataFrame({
        "tanggal": j["daily"]["time"],
        "suhu_maks_C": j["daily"]["temperature_2m_max"],
        "suhu_min_C": j["daily"]["temperature_2m_min"],
        "curah_hujan_mm": j["daily"]["precipitation_sum"]
    })
    df_api.to_csv("data/raw/cuaca_bali.csv", index=False)
    print(f"✅ Data API: {len(df_api)} baris")
    print(df_api.head())
else:
    print("❌ Gagal mendapatkan data dari API setelah 3 kali percobaan.")

Status: 200
✅ Data API: 8 baris
      tanggal  suhu_maks_C  suhu_min_C  curah_hujan_mm
0  2026-11-19         30.1        24.2             7.2
1  2026-11-20         28.5        24.7             8.8
2  2026-11-21         29.5        23.3             9.5
3  2026-11-22         29.1        24.2             4.9
4  2026-11-23         29.1        24.7             7.0


In [10]:
print("=== Cek robots.txt ===")
r = requests.get("https://id.wikipedia.org/robots.txt", timeout=10)
print(r.text[:600], "...\n")

URL_SCRAPE = "https://id.wikipedia.org/wiki/Pariwisata_di_Bali"
headers = {"User-Agent": "email_kampus@student.unud.ac.id"}
time.sleep(2)

res = requests.get(URL_SCRAPE, headers=headers, timeout=30)
print(f"Status: {res.status_code}")

with open("data/raw/wisata_bali.html", "w", encoding="utf-8") as f:
    f.write(res.text)

soup = BeautifulSoup(res.text, "html.parser")
tabel = soup.find("table", {"class": "wikitable"})

if tabel:
    df_scrape = pd.read_html(str(tabel))[0]
    df_scrape.to_csv("data/raw/wisata_bali.csv", index=False)
    print(f"✅ Scraping: {df_scrape.shape[0]} baris")
    display(df_scrape.head())
else:
    print("⚠️ Tabel tidak ditemukan — ganti halaman lain")


=== Cek robots.txt ===
Please set a user-agent and respect our robot policy https://w.wiki/4wJS. See also https://phabricator.wikimedia.org/T400119.
 ...

Status: 200
✅ Scraping: 6 baris


/tmp/ipykernel_3058/1237284324.py:19: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  df_scrape = pd.read_html(str(tabel))[0]


,Tahun,Jumlah Kunjungan,Perbandingan (%)
0,2010,2493058,tidak ada data
1,2011,2756579,+10.57%
2,2012,2892019,+4.91%
3,2013,3278598,+13.37%
4,2014,3766638,+14.89%


In [11]:
df = pd.read_csv("data/raw/kunjungan_wisata.csv")

print("="*50)
print("📋 IDENTIFIKASI DATASET")
print("="*50)
print(f"Jenis Data       : Terstruktur")
print(f"Jumlah Baris     : {df.shape[0]}")
print(f"Jumlah Kolom     : {df.shape[1]}")
print(f"\nTipe Data:\n{df.dtypes}")
print(f"\nSatuan           : Jumlah wisatawan = orang")
print(f"Cakupan Wilayah  : Provinsi Bali")
print(f"Cakupan Waktu    : 2018–2025")

# Estimasi storage
ukuran_mb = os.path.getsize("data/raw/kunjungan_wisata.csv") / (1024*1024)
tahunan = ukuran_mb * 12
cadangan = tahunan * 0.3
total = tahunan + cadangan

print(f"\n📦 ESTIMASI PENYIMPANAN")
print(f"Sekarang         : {ukuran_mb:.4f} MB")
print(f"12 Bulan         : {tahunan:.4f} MB")
print(f"+ Cadangan 30%    : {cadangan:.4f} MB")
print(f"TOTAL            : {total:.4f} MB")
print(f"Asumsi: ukuran bertambah konsisten tiap bulan")


📋 IDENTIFIKASI DATASET
Jenis Data       : Terstruktur
Jumlah Baris     : 96
Jumlah Kolom     : 3

Tipe Data:
tahun                int64
bulan               object
jumlah_wisatawan     int64
dtype: object

Satuan           : Jumlah wisatawan = orang
Cakupan Wilayah  : Provinsi Bali
Cakupan Waktu    : 2018–2025

📦 ESTIMASI PENYIMPANAN
Sekarang         : 0.0015 MB
12 Bulan         : 0.0179 MB
+ Cadangan 30%    : 0.0054 MB
TOTAL            : 0.0232 MB
Asumsi: ukuran bertambah konsisten tiap bulan


In [12]:
checklist = """
# ✅ DATA ETHICS CHECKLIST — 9 BUTIR

| No | Pertanyaan | Jawaban | Bukti |
|----|-----------|---------|-------|
| 1 | Lisensi diperiksa? | ✅ Ya | CC-BY 4.0 / CC BY-SA 3.0 |
| 2 | Sesuai ketentuan? | ✅ Ya | Hanya tugas akademik |
| 3 | Ada data PII? | ❌ Tidak | Data agregat |
| 4 | Ada quasi-identifier? | ❌ Tidak | Tidak ada identitas |
| 5 | Provenance lengkap? | ✅ Ya | Lihat PROVENANCE.md |
| 6 | Situs terlarang? | ✅ Tidak | Sesuai aturan soal |
| 7 | API key aman? | ✅ Ya | Tanpa token |
| 8 | Patuh robots.txt? | ✅ Ya | Sudah diperiksa |
| 9 | Siap hapus data? | ✅ Ya | Atas permintaan pemilik |
"""

with open("data/raw/ETIKA_DATA.md", "w", encoding="utf-8") as f:
    f.write(checklist)
print(checklist)



# ✅ DATA ETHICS CHECKLIST — 9 BUTIR

| No | Pertanyaan | Jawaban | Bukti |
|----|-----------|---------|-------|
| 1 | Lisensi diperiksa? | ✅ Ya | CC-BY 4.0 / CC BY-SA 3.0 |
| 2 | Sesuai ketentuan? | ✅ Ya | Hanya tugas akademik |
| 3 | Ada data PII? | ❌ Tidak | Data agregat |
| 4 | Ada quasi-identifier? | ❌ Tidak | Tidak ada identitas |
| 5 | Provenance lengkap? | ✅ Ya | Lihat PROVENANCE.md |
| 6 | Situs terlarang? | ✅ Tidak | Sesuai aturan soal |
| 7 | API key aman? | ✅ Ya | Tanpa token |
| 8 | Patuh robots.txt? | ✅ Ya | Sudah diperiksa |
| 9 | Siap hapus data? | ✅ Ya | Atas permintaan pemilik |



In [13]:
# Baca data
df1 = pd.read_csv("data/raw/kunjungan_wisata.csv")
df2 = pd.read_csv("data/raw/cuaca_bali.csv")

# Siapkan kolom kunci
df2["tanggal"] = pd.to_datetime(df2["tanggal"])
df2["tahun"] = df2["tanggal"].dt.year
df2["bulan"] = df2["tanggal"].dt.strftime("%b")

# Agregat cuaca per bulan
df2_agg = df2.groupby(["tahun", "bulan"]).agg({
    "suhu_maks_C": "mean",
    "suhu_min_C": "mean",
    "curah_hujan_mm": "sum"
}).reset_index()

# Gabungkan
df_gabung = pd.merge(df1, df2_agg, on=["tahun", "bulan"], how="inner", validate="1:1")

# Cek duplikasi
duplikat = df_gabung.duplicated().sum()
print(f"Baris duplikat: {duplikat}")
print(f"Hasil akhir: {df_gabung.shape[0]} baris, {df_gabung.shape[1]} kolom")

# Simpan
df_gabung.to_csv("data/hasil_gabungan.csv", index=False)
print("✅ Hasil disimpan: data/hasil_gabungan.csv")
df_gabung.head()


Baris duplikat: 0
Hasil akhir: 0 baris, 6 kolom
✅ Hasil disimpan: data/hasil_gabungan.csv


,tahun,bulan,jumlah_wisatawan,suhu_maks_C,suhu_min_C,curah_hujan_mm
